# Estudio de Ablación - Detección de Neumonía
### Cuantifica la contribución individual de cada componente del pipeline

Responde a la crítica: *"no se cuantifica la contribución individual de cada componente"*.

Se entrena **DenseNet121** (arquitectura representativa) bajo 4 configuraciones:

| # | Configuración | Transfer Learning | Aumentación | Qué mide |
|---|---|---|---|---|
| A | Completa (baseline) | ✓ ImageNet | ✓ Sí | Referencia |
| B | Sin transfer learning | ✗ Desde cero | ✓ Sí | Aporte de ImageNet |
| C | Sin aumentación | ✓ ImageNet | ✗ No | Aporte de la aumentación |
| D | Sin ensamblado | — | — | (ya calculado en el experimento principal) |

**Tiempo estimado:** 30-45 min en GPU T4 (solo 3 entrenamientos de 1 modelo).

**Instrucciones:** GPU T4, ejecuta en orden, sube kaggle.json.

In [ ]:
!pip install -q kaggle timm scikit-learn
import torch
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.is_available())

In [ ]:
from google.colab import files
import os
print('Sube tu kaggle.json:')
uploaded = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
for fn in uploaded.keys(): os.rename(fn, '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /content/data --unzip
print('Dataset listo')

In [ ]:
# Misma particion binaria 70/15/15, semilla 42 (identica al experimento principal)
import shutil, random
from pathlib import Path
random.seed(42)
SRC=Path('/content/data/chest_xray')
ALL={'NORMAL':[],'PNEUMONIA':[]}
for split in ['train','test','val']:
    sd=SRC/split
    if not sd.exists(): continue
    for img in (sd/'NORMAL').glob('*.jpeg'): ALL['NORMAL'].append(img)
    for img in (sd/'PNEUMONIA').glob('*.jpeg'): ALL['PNEUMONIA'].append(img)
DEST=Path('/content/dataset_bin')
if DEST.exists(): shutil.rmtree(DEST)
for cls,imgs in ALL.items():
    imgs=imgs.copy(); random.shuffle(imgs)
    n=len(imgs); ntr=int(n*0.70); nv=int(n*0.15)
    for sn,si in {'train':imgs[:ntr],'val':imgs[ntr:ntr+nv],'test':imgs[ntr+nv:]}.items():
        od=DEST/sn/cls; od.mkdir(parents=True,exist_ok=True)
        for img in si: shutil.copy(img,od/img.name)
print('Particion lista (misma semilla que el experimento principal)')
for sn in ['train','val','test']:
    print(f'  {sn}:', {c: len(list((DEST/sn/c).glob("*"))) for c in ALL})

In [ ]:
import timm, numpy as np, time, json
import torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_CLASSES=2; BATCH_SIZE=32; EPOCHS=10; LR=1e-4; PATIENCE=4; IMG=224

def get_loaders(use_augmentation=True):
    """Si use_augmentation=False, el train no recibe aumentacion (solo resize+normalize)."""
    if use_augmentation:
        train_tf=transforms.Compose([
            transforms.Resize((IMG,IMG)), transforms.RandomHorizontalFlip(0.5),
            transforms.RandomRotation(10), transforms.ColorJitter(brightness=0.15,contrast=0.15),
            transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    else:
        train_tf=transforms.Compose([
            transforms.Resize((IMG,IMG)), transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    eval_tf=transforms.Compose([transforms.Resize((IMG,IMG)),transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    tr=datasets.ImageFolder('/content/dataset_bin/train',transform=train_tf)
    va=datasets.ImageFolder('/content/dataset_bin/val',transform=eval_tf)
    te=datasets.ImageFolder('/content/dataset_bin/test',transform=eval_tf)
    return (DataLoader(tr,batch_size=BATCH_SIZE,shuffle=True,num_workers=2),
            DataLoader(va,batch_size=BATCH_SIZE,shuffle=False,num_workers=2),
            DataLoader(te,batch_size=BATCH_SIZE,shuffle=False,num_workers=2), tr.classes)

def train_and_eval(use_pretrained, use_augmentation, config_name):
    """Entrena DenseNet121 con/sin transfer learning y con/sin aumentacion."""
    torch.manual_seed(42); np.random.seed(42)
    tr,va,te,classes=get_loaders(use_augmentation)
    model=timm.create_model('densenet121',pretrained=use_pretrained,num_classes=NUM_CLASSES).to(device)
    criterion=nn.CrossEntropyLoss()
    optimizer=optim.AdamW(model.parameters(),lr=LR,weight_decay=0.01)
    scheduler=optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS)
    best_acc,best_state,no_imp=0.0,None,0
    print(f'\n{"="*55}\n{config_name}\n  pretrained={use_pretrained} | augmentation={use_augmentation}\n{"="*55}')
    for ep in range(EPOCHS):
        t0=time.time(); model.train()
        for x,y in tr:
            x,y=x.to(device),y.to(device)
            optimizer.zero_grad(); loss=criterion(model(x),y); loss.backward(); optimizer.step()
        scheduler.step(); model.eval(); vp,vt=[],[]
        with torch.no_grad():
            for x,y in va:
                out=model(x.to(device)); vp.extend(out.argmax(1).cpu().numpy()); vt.extend(y.numpy())
        vacc=accuracy_score(vt,vp)
        print(f'  Ep {ep+1}/{EPOCHS} | val_acc={vacc:.4f} | {time.time()-t0:.0f}s')
        if vacc>best_acc: best_acc=vacc; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; no_imp=0
        else:
            no_imp+=1
            if no_imp>=PATIENCE: print(f'  Early stopping ep {ep+1}'); break
    if best_state: model.load_state_dict(best_state)
    # Evaluacion en test
    model.eval(); preds,labels=[],[]
    with torch.no_grad():
        for x,y in te:
            out=model(x.to(device)); preds.extend(out.argmax(1).cpu().numpy()); labels.extend(y.numpy())
    preds,labels=np.array(preds),np.array(labels)
    acc=accuracy_score(labels,preds); f1=f1_score(labels,preds,average='macro',zero_division=0)
    prec=precision_score(labels,preds,average='macro',zero_division=0)
    rec=recall_score(labels,preds,average='macro',zero_division=0)
    cm=confusion_matrix(labels,preds); tn,fp=cm[0,0],cm[0,1]
    spec=tn/(tn+fp) if (tn+fp)>0 else 0
    print(f'  >> TEST: acc={acc*100:.2f}% f1={f1*100:.2f}%')
    del model; torch.cuda.empty_cache()
    return {'accuracy':acc,'precision':prec,'recall_sensitivity':rec,'f1':f1,
            'specificity':float(spec),'confusion_matrix':cm.tolist()}

## Ejecutar las 3 configuraciones de ablación

In [ ]:
ablacion = {}

# A: Configuración completa (baseline) - transfer learning + aumentación
ablacion['A. Completa (TL + Aumentación)'] = train_and_eval(
    use_pretrained=True, use_augmentation=True,
    config_name='A. CONFIGURACIÓN COMPLETA (baseline)')

# B: Sin transfer learning (entrenar desde cero)
ablacion['B. Sin Transfer Learning'] = train_and_eval(
    use_pretrained=False, use_augmentation=True,
    config_name='B. SIN TRANSFER LEARNING (desde cero)')

# C: Sin aumentación de datos
ablacion['C. Sin Aumentación'] = train_and_eval(
    use_pretrained=True, use_augmentation=False,
    config_name='C. SIN AUMENTACIÓN DE DATOS')

print('\n\nAblación completa.')

In [ ]:
# Tabla de ablación con la contribución de cada componente
import pandas as pd
baseline_acc = ablacion['A. Completa (TL + Aumentación)']['accuracy']

rows=[]
for name,m in ablacion.items():
    delta = (m['accuracy'] - baseline_acc)*100
    rows.append({'Configuración':name,
                 'Exactitud (%)':round(m['accuracy']*100,2),
                 'F1 (%)':round(m['f1']*100,2),
                 'Sensibilidad (%)':round(m['recall_sensitivity']*100,2),
                 'Especificidad (%)':round(m['specificity']*100,2),
                 'Δ vs. completa':('—' if abs(delta)<0.001 else f'{delta:+.2f}')})
df=pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_csv('/content/ablacion_resultados.csv',index=False)
with open('/content/ablacion_resumen.json','w') as f: json.dump(ablacion,f,indent=2)

print('\n=== CONTRIBUCIÓN DE CADA COMPONENTE ===')
tl_contrib = (baseline_acc - ablacion['B. Sin Transfer Learning']['accuracy'])*100
aug_contrib = (baseline_acc - ablacion['C. Sin Aumentación']['accuracy'])*100
print(f'  Transfer Learning aporta: {tl_contrib:+.2f} puntos de exactitud')
print(f'  Aumentación de datos aporta: {aug_contrib:+.2f} puntos de exactitud')
print('  (El aporte del ensamblado ya se cuantificó en el experimento principal)')

files.download('/content/ablacion_resultados.csv')
files.download('/content/ablacion_resumen.json')
print('\nListo. Pasame los 2 archivos.')